# Weather Data Analysis and Rainfall Prediction Using Machine Learning
**Dataset:** WeatherAUS (Rattle / Kaggle) · **Target:** `RainTomorrow` (Yes/No) · **Run order:** top to bottom (Runtime → Run all).

Every number below is computed from the real data when the notebook runs. Nothing is hard-coded.

## 1. Install / import libraries and load the dataset

In [ ]:
import os, json, time, warnings, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import joblib, sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, balanced_accuracy_score, confusion_matrix,
                             classification_report, precision_recall_curve, roc_curve)
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); rng = np.random.default_rng(SEED)      # reproducibility
os.makedirs("outputs", exist_ok=True)                          # all graphs are saved here
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

def save_fig(name):
    plt.tight_layout(); plt.savefig(f"outputs/{name}.png", dpi=100, bbox_inches="tight"); plt.show(); plt.close("all")

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
# Load the data: local file -> download -> manual upload (practical fallbacks if the URL fails)
DATA_URL = "https://rattle.togaware.com/weatherAUS.csv"
candidates = ["weatherAUS.csv", "data/weatherAUS.csv", "/content/weatherAUS.csv", "/mnt/user-data/uploads/weatherAUS.csv"]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    try:
        urllib.request.urlretrieve(DATA_URL, "weatherAUS.csv"); path = "weatherAUS.csv"
    except Exception as e:
        print("Download failed:", e)
        try:
            from google.colab import files            # Colab fallback: upload the Kaggle CSV by hand
            up = files.upload(); path = list(up)[0]
        except ImportError:
            raise FileNotFoundError("Put weatherAUS.csv next to this notebook (Kaggle link in README.md).")
df_raw = pd.read_csv(path)
print("Loaded:", path, "->", df_raw.shape)

## 2. First look: head, tail, shape, columns, dtypes, info, missing values, duplicates

In [ ]:
display(df_raw.head()); display(df_raw.tail())
print("Shape:", df_raw.shape); print("Columns:", list(df_raw.columns)); print("\nDtypes:"); print(df_raw.dtypes)
print("\nInfo:"); df_raw.info()
miss = pd.DataFrame({"missing": df_raw.isna().sum(), "missing_%": (df_raw.isna().mean()*100).round(2)})
print("\nMissing values:"); display(miss[miss.missing > 0].sort_values("missing", ascending=False))
print("Duplicate rows:", df_raw.duplicated().sum())
print("\nDate range:", df_raw.Date.min(), "->", df_raw.Date.max(), "| locations:", df_raw.Location.nunique())
print("\nTarget distribution (incl. missing):"); print(df_raw.RainTomorrow.value_counts(dropna=False))

## 3. Data cleaning and preparation
* `Date` → datetime; exact duplicates dropped; rows **without a target** dropped (cannot be used for supervised learning).
* **`RISK_MM` removed**: it is the *next day's* rainfall amount, i.e. the target in disguise (target leakage).
* Remaining features are all observations of day *t* (9 am / 3 pm readings, today's rainfall), so a forecast for day *t+1* can legitimately use them.
* `df` keeps NaNs on purpose: the ML pipeline imputes them **using training data only** (no preprocessing leakage).
* `df_clean` (median-by-location imputation, binary/label encodings) is for EDA and for the saved `cleaned_weather.csv` only.

In [ ]:
df = df_raw.copy()
df["Date"] = pd.to_datetime(df["Date"])
n0 = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["RainTomorrow"])
leaky = [c for c in ["RISK_MM"] if c in df.columns]
df = df.drop(columns=leaky)
df = df.sort_values(["Date", "Location"]).reset_index(drop=True)      # chronological order
df["Year"], df["Month"] = df.Date.dt.year, df.Date.dt.month
df["Season"] = df.Month.map({12:"Summer",1:"Summer",2:"Summer",3:"Autumn",4:"Autumn",5:"Autumn",
                             6:"Winter",7:"Winter",8:"Winter",9:"Spring",10:"Spring",11:"Spring"})
df["RainTomorrow_bin"] = (df.RainTomorrow == "Yes").astype(int)
df["RainToday_bin"] = df.RainToday.map({"No": 0, "Yes": 1})
print(f"Rows: {n0} -> {len(df)} | removed leaky columns: {leaky} | 'RISK_MM' still present? {'RISK_MM' in df.columns}")

NUM = ["MinTemp","MaxTemp","Rainfall","Evaporation","Sunshine","WindGustSpeed","WindSpeed9am","WindSpeed3pm",
       "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm","Cloud9am","Cloud3pm","Temp9am","Temp3pm"]
WIND = ["WindGustDir", "WindDir9am", "WindDir3pm"]

# ---- cleaned copy for EDA / export (imputation here is NOT used for model evaluation) ----
df_clean = df.copy()
df_clean[NUM] = df_clean.groupby("Location")[NUM].transform(lambda s: s.fillna(s.median()))
df_clean[NUM] = df_clean[NUM].fillna(df_clean[NUM].median())                 # locations that never measure a variable
for c in WIND + ["RainToday"]:
    df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])
df_clean["RainToday_bin"] = (df_clean.RainToday == "Yes").astype(int)
for c in ["Location"] + WIND:                                                 # label encoding of categoricals
    df_clean[c + "_code"] = df_clean[c].astype("category").cat.codes
df_clean.to_csv("cleaned_weather.csv", index=False)
print("Saved cleaned_weather.csv", df_clean.shape, "| remaining NaNs in df_clean:", int(df_clean.isna().sum().sum()))

## 4. Exploratory Data Analysis (graphs saved in `outputs/`)

In [ ]:
# 4a Histograms
cols = ["MinTemp","MaxTemp","Humidity3pm","Pressure3pm","WindGustSpeed","Temp9am"]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for a, c in zip(ax.ravel(), cols):
    sns.histplot(df[c].dropna(), bins=40, kde=True, ax=a, color="steelblue"); a.set_title(f"Histogram of {c}")
save_fig("01_histograms")

# 4b Box plots: key variables split by tomorrow's rain
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, c in zip(ax, ["Humidity3pm", "Pressure3pm", "Sunshine", "MaxTemp"]):
    sns.boxplot(data=df, x="RainTomorrow", y=c, ax=a, palette="Set2"); a.set_title(f"{c} vs RainTomorrow")
save_fig("02_boxplots")

In [ ]:
# 4c Bar charts: rain probability by location and by month
fig, ax = plt.subplots(1, 2, figsize=(16, 9), gridspec_kw={"width_ratios": [1.3, 1]})
loc_rate = df.groupby("Location").RainTomorrow_bin.mean().sort_values()
loc_rate.plot.barh(ax=ax[0], color="teal"); ax[0].set_title("P(RainTomorrow = Yes) by location"); ax[0].set_xlabel("proportion")
df.groupby("Month").RainTomorrow_bin.mean().plot.bar(ax=ax[1], color="coral"); ax[1].set_title("P(RainTomorrow = Yes) by month"); ax[1].set_ylabel("proportion")
save_fig("03_bar_charts")

# 4d Pie charts
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, (c, t) in zip(ax, [("RainTomorrow", "RainTomorrow"), ("RainToday", "RainToday"), ("Season", "Rows by season")]):
    df[c].value_counts().plot.pie(autopct="%1.1f%%", ax=a, startangle=90); a.set_ylabel(""); a.set_title(t)
save_fig("04_pie_charts")

In [ ]:
# 4e Scatter plots (random 6000-row sample for readability)
s = df.sample(6000, random_state=SEED)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=s, x="Humidity3pm", y="Temp3pm", hue="RainTomorrow", alpha=.5, s=14, ax=ax[0])
sns.scatterplot(data=s, x="Pressure3pm", y="Humidity3pm", hue="RainTomorrow", alpha=.5, s=14, ax=ax[1])
save_fig("05_scatter_plots")

# 4f Line chart: monthly averages over time
m = df.set_index("Date")[["Rainfall", "MaxTemp", "RainTomorrow_bin"]].resample("MS").mean()
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for a, (c, col) in zip(ax, [("MaxTemp", "tomato"), ("Rainfall", "royalblue"), ("RainTomorrow_bin", "seagreen")]):
    m[c].plot(ax=a, color=col); a.set_ylabel(c)
ax[0].set_title("Monthly average over all stations")
save_fig("06_line_chart")

# 4g Correlation heatmap (pairwise-complete observations)
plt.figure(figsize=(12, 9))
sns.heatmap(df[NUM + ["RainTomorrow_bin"]].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation heatmap"); save_fig("07_correlation_heatmap")
print(df[NUM].corrwith(df.RainTomorrow_bin).sort_values(ascending=False).round(3).to_string())

## 5. Descriptive statistics

In [ ]:
def iqr_outliers(x):
    q1, q3 = x.quantile([.25, .75]); i = q3 - q1
    return int(((x < q1 - 1.5*i) | (x > q3 + 1.5*i)).sum())
desc = pd.DataFrame({
    "mean": df[NUM].mean(), "median": df[NUM].median(), "mode": [df[c].mode().iloc[0] for c in NUM],
    "variance": df[NUM].var(), "std": df[NUM].std(), "min": df[NUM].min(), "max": df[NUM].max(),
    "Q1": df[NUM].quantile(.25), "Q2": df[NUM].quantile(.5), "Q3": df[NUM].quantile(.75)})
desc["IQR"] = desc.Q3 - desc.Q1
desc["IQR_outliers"] = [iqr_outliers(df[c].dropna()) for c in NUM]
display(desc.round(3))
print("\ndescribe():"); display(df[NUM].describe().T.round(3))

## 6. Probability distributions (real data vs simulated)

In [ ]:
# --- Normal: real Pressure3pm vs a simulated Normal(mu, sd) ---
x = df["Pressure3pm"].dropna(); mu, sd = x.mean(), x.std()
sim_n = rng.normal(mu, sd, len(x))
print(f"Pressure3pm: mean={mu:.2f}, sd={sd:.2f}, skew={stats.skew(x):.3f}, excess kurtosis={stats.kurtosis(x):.3f}")
print("Share within 1/2/3 sd (real):", [round(((x > mu-k*sd) & (x < mu+k*sd)).mean(), 4) for k in (1, 2, 3)], "| theory: [0.6827, 0.9545, 0.9973]")
sh = stats.shapiro(x.sample(5000, random_state=SEED)); print(f"Shapiro-Wilk (n=5000 sample): W={sh.statistic:.4f}, p={sh.pvalue:.3g}")
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5)); g = np.linspace(x.min(), x.max(), 300)
for a, d, t in [(ax[0], x, "Real Pressure3pm"), (ax[1], sim_n, "Simulated Normal")]:
    sns.histplot(d, bins=50, stat="density", ax=a, color="lightsteelblue"); a.plot(g, stats.norm.pdf(g, mu, sd), "r"); a.set_title(t)
stats.probplot(x.sample(5000, random_state=SEED), dist="norm", plot=ax[2]); ax[2].set_title("Q-Q plot (real)")
save_fig("08_normal_distribution")

In [ ]:
# --- Uniform: simulated U(lo, hi) vs real MaxTemp (KS test against a uniform) ---
y = df["MaxTemp"].dropna(); lo, hi = y.min(), y.max()
sim_u = rng.uniform(lo, hi, len(y))
ks_sim = stats.kstest(sim_u, "uniform", args=(lo, hi - lo)); ks_real = stats.kstest(y, "uniform", args=(lo, hi - lo))
print(f"KS vs Uniform -> simulated: D={ks_sim.statistic:.4f}, p={ks_sim.pvalue:.3g} | real MaxTemp: D={ks_real.statistic:.4f}, p={ks_real.pvalue:.3g}")
cnt = df.Month.value_counts().sort_index(); chi = stats.chisquare(cnt)
print(f"Rows per month vs uniform: chi2={chi.statistic:.1f}, p={chi.pvalue:.3g} (observations are roughly, not perfectly, uniform in time)")
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
sns.histplot(sim_u, bins=40, ax=ax[0], color="orchid").set_title("Simulated Uniform"); sns.histplot(y, bins=40, ax=ax[1], color="gold").set_title("Real MaxTemp (not uniform)")
cnt.plot.bar(ax=ax[2], color="gray"); ax[2].set_title("Observations per month")
save_fig("09_uniform_distribution")

# --- Binomial: rainy days in blocks of 7 days at Sydney vs Binomial(7, p) ---
r = df[df.Location == "Sydney"].sort_values("Date").RainTomorrow_bin.values
p_hat = r.mean(); blocks = len(r) // 7; k = r[:blocks*7].reshape(blocks, 7).sum(1)
obs = np.bincount(k, minlength=8) / blocks; theo = stats.binom.pmf(np.arange(8), 7, p_hat); sim_b = np.bincount(rng.binomial(7, p_hat, blocks), minlength=8) / blocks
print(f"Sydney: p(rain day)={p_hat:.3f}, blocks={blocks} | observed var={k.var(ddof=1):.3f} vs binomial var=np(1-p)={7*p_hat*(1-p_hat):.3f}")
print("-> variance above the binomial value = rain days cluster (days are not independent).")
w = .27; plt.figure(figsize=(9, 4.5)); plt.bar(np.arange(8)-w, obs, w, label="Observed (real)"); plt.bar(np.arange(8), theo, w, label="Binomial pmf"); plt.bar(np.arange(8)+w, sim_b, w, label="Simulated")
plt.xlabel("rainy days in a 7-day block"); plt.ylabel("probability"); plt.title("Binomial model for weekly rainy days (Sydney)"); plt.legend()
save_fig("10_binomial_distribution")

## 7. Sampling techniques (population = all labelled rows)

In [ ]:
N, n = len(df), 2000
srs = df.sample(n=n, random_state=SEED)                                            # simple random
step = N // n; start = int(rng.integers(0, step)); systematic = df.iloc[start::step]  # systematic: every k-th row
stratified = df.groupby("Location", group_keys=False).sample(frac=n/N, random_state=SEED)  # proportional strata = Location
clusters = rng.choice(df.Location.unique(), size=5, replace=False)                 # location-based cluster sampling:
cluster = df[df.Location.isin(clusters)]                                            # whole clusters (stations) are selected
print(f"Population N={N} | SRS n={len(srs)} | systematic (k={step}, start={start}) n={len(systematic)} | stratified n={len(stratified)} | cluster n={len(cluster)}")
print("Selected clusters:", list(clusters))

## 8. Sample vs population statistics

In [ ]:
vars_ = ["MaxTemp", "Humidity3pm", "Rainfall", "RainTomorrow_bin"]
groups = {"Population": df, "Simple random": srs, "Systematic": systematic, "Stratified": stratified, "Cluster": cluster}
tab = pd.DataFrame({name: [d[v].mean() for v in vars_] + [d[v].std() for v in vars_] + [len(d)] for name, d in groups.items()},
                   index=[f"mean {v}" for v in vars_] + [f"std {v}" for v in vars_] + ["n"]).T
display(tab.round(3))
err = pd.DataFrame({name: [abs(d[v].mean() - df[v].mean()) / abs(df[v].mean()) * 100 for v in vars_] for name, d in list(groups.items())[1:]}, index=vars_)
print("\nAbsolute % error of the sample mean relative to the population mean:"); display(err.round(2))
err.T.plot.bar(figsize=(10, 4.5), rot=0); plt.ylabel("% error of mean"); plt.title("Sampling methods: error vs population")
save_fig("11_sampling_error")

## 9. Central Limit Theorem (population = Rainfall, strongly right-skewed)

In [ ]:
pop = df["Rainfall"].dropna().values; mu, sigma = pop.mean(), pop.std()
print(f"Population: N={len(pop)}, mean={mu:.3f}, sd={sigma:.3f}, skew={stats.skew(pop):.2f}")
sizes, reps, rows = [10, 30, 50, 100], 3000, []
fig, ax = plt.subplots(2, 3, figsize=(16, 8)); ax = ax.ravel()
ax[0].hist(np.clip(pop, 0, np.percentile(pop, 99)), bins=50, color="gray"); ax[0].set_title("Population Rainfall (clipped at 99th pct)")
for a, s in zip(ax[1:5], sizes):
    means = rng.choice(pop, size=(reps, s)).mean(axis=1)                      # `reps` samples of size s
    se_emp, se_th = means.std(ddof=1), sigma / np.sqrt(s)
    rows.append({"n": s, "mean of sample means": means.mean(), "empirical SE": se_emp, "theoretical SE (sigma/sqrt n)": se_th, "skew of sampling dist": stats.skew(means)})
    sns.histplot(means, bins=40, stat="density", kde=True, ax=a, color="skyblue")
    xs = np.linspace(means.min(), means.max(), 200); a.plot(xs, stats.norm.pdf(xs, mu, se_th), "r"); a.set_title(f"Sample means, n={s}")
clt = pd.DataFrame(rows); display(clt.round(4))
ax[5].plot(clt.n, clt["empirical SE"], "o-", label="empirical"); ax[5].plot(clt.n, clt["theoretical SE (sigma/sqrt n)"], "s--", label="sigma/sqrt(n)")
ax[5].set_xlabel("sample size n"); ax[5].set_ylabel("standard error"); ax[5].legend(); ax[5].set_title("Standard error shrinks with n")
save_fig("12_central_limit_theorem")

## 10. Covariance, Pearson and Spearman correlation

In [ ]:
pairs = [("MinTemp", "MaxTemp"), ("Humidity9am", "Humidity3pm"), ("Temp3pm", "Humidity3pm"), ("Pressure3pm", "Humidity3pm"), ("Sunshine", "Humidity3pm"), ("Pressure9am", "Pressure3pm")]
rows = []
for a, b in pairs:
    d = df[[a, b]].dropna(); pr = stats.pearsonr(d[a], d[b]); sp = stats.spearmanr(d[a], d[b])
    rows.append({"pair": f"{a} vs {b}", "n": len(d), "covariance": d[a].cov(d[b]), "Pearson r": pr[0], "Pearson p": pr[1], "Spearman rho": sp[0], "Spearman p": sp[1]})
corr_tab = pd.DataFrame(rows); display(corr_tab.round(4))
print("p-values printed as 0 are below floating-point precision (very large n).")
fig, ax = plt.subplots(2, 3, figsize=(16, 9)); samp = df.sample(5000, random_state=SEED)
for i, (a_, (a, b)) in enumerate(zip(ax.ravel(), pairs)):
    sns.regplot(data=samp, x=a, y=b, ax=a_, scatter_kws={"s": 6, "alpha": .3}, line_kws={"color": "red"})
    a_.set_title(f"{a} vs {b}  (r={corr_tab['Pearson r'].iloc[i]:.2f})")
save_fig("13_correlation_scatter")

## 11. Confidence intervals and hypothesis tests (alpha = 0.05)

In [ ]:
ALPHA = 0.05
def report(title, H0, H1, stat_name, stat, p, extra=""):
    dec = "Reject H0" if p < ALPHA else "Fail to reject H0"
    interp = "The data give statistically significant evidence for H1." if p < ALPHA else "There is not enough evidence to support H1."
    print(f"\n=== {title} ===\nH0: {H0}\nH1: {H1}\n{stat_name} = {stat:.4f} | p-value = {p:.4g} | alpha = {ALPHA}\nDecision: {dec}\nInterpretation: {interp} {extra}")

# ---- Confidence intervals ----
v = df["MaxTemp"].dropna(); s100 = v.sample(100, random_state=SEED)
ci100 = stats.t.interval(0.95, len(s100)-1, loc=s100.mean(), scale=stats.sem(s100))
ci_all = stats.t.interval(0.95, len(v)-1, loc=v.mean(), scale=stats.sem(v))
print(f"95% CI for mean MaxTemp, sample n=100 : ({ci100[0]:.3f}, {ci100[1]:.3f}) | contains population mean {v.mean():.3f}? {ci100[0] <= v.mean() <= ci100[1]}")
print(f"95% CI for mean MaxTemp, all n={len(v)}: ({ci_all[0]:.3f}, {ci_all[1]:.3f})  <- narrower: SE falls with n")
pr_, nn = df.RainTomorrow_bin.mean(), len(df); se_p = np.sqrt(pr_*(1-pr_)/nn)
print(f"95% CI for P(RainTomorrow=Yes): ({pr_-1.96*se_p:.4f}, {pr_+1.96*se_p:.4f}) (point estimate {pr_:.4f})")

In [ ]:
# ---- One-sample t-test: Sydney MaxTemp vs benchmark mu0 = 22 C (benchmark chosen by us for demonstration) ----
syd = df[df.Location == "Sydney"].MaxTemp.dropna(); mu0 = 22
print(f"Assumption check (Sydney MaxTemp, n={len(syd)}): skew={stats.skew(syd):.2f}; n is large, so the t-test is robust (CLT)")
t = stats.ttest_1samp(syd, mu0)
report("One-sample t-test", f"mean Sydney MaxTemp = {mu0}", f"mean Sydney MaxTemp != {mu0}", "t", t.statistic, t.pvalue, f"(sample mean = {syd.mean():.3f})")

# ---- Independent (Welch) t-test: Humidity3pm, rainy tomorrow vs dry tomorrow ----
h_yes = df[df.RainTomorrow == "Yes"].Humidity3pm.dropna(); h_no = df[df.RainTomorrow == "No"].Humidity3pm.dropna()
lev = stats.levene(h_yes, h_no); print(f"\nLevene equal-variance test: W={lev.statistic:.2f}, p={lev.pvalue:.3g} -> Welch's t-test (no equal-variance assumption) is used")
t = stats.ttest_ind(h_yes, h_no, equal_var=False)
sp_ = np.sqrt(((len(h_yes)-1)*h_yes.var() + (len(h_no)-1)*h_no.var()) / (len(h_yes)+len(h_no)-2)); d_ = (h_yes.mean() - h_no.mean()) / sp_
mw = stats.mannwhitneyu(h_yes, h_no)
report("Independent t-test (Welch)", "mean Humidity3pm (rain tomorrow) = mean Humidity3pm (no rain)", "the two means differ", "t", t.statistic, t.pvalue,
       f"(means {h_yes.mean():.2f} vs {h_no.mean():.2f}; Cohen's d = {d_:.2f}; Mann-Whitney p = {mw.pvalue:.3g})")

In [ ]:
# ---- Paired t-test: Temp9am vs Temp3pm on the same day/station ----
pt = df[["Temp9am", "Temp3pm"]].dropna(); diff = pt.Temp3pm - pt.Temp9am
sh = stats.shapiro(diff.sample(5000, random_state=SEED))
print(f"Assumption check (differences, n={len(diff)}): skew={stats.skew(diff):.2f}, Shapiro p (5000 sample)={sh.pvalue:.3g}; large n -> t-test still reasonable")
t = stats.ttest_rel(pt.Temp3pm, pt.Temp9am); wil = stats.wilcoxon(diff.sample(5000, random_state=SEED))
report("Paired t-test", "mean(Temp3pm - Temp9am) = 0", "mean(Temp3pm - Temp9am) != 0", "t", t.statistic, t.pvalue, f"(mean difference = {diff.mean():.3f} C; Wilcoxon p = {wil.pvalue:.3g})")

# ---- One-way ANOVA: Humidity3pm across five capital cities ----
want = ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"]
cities = [c for c in want if c in set(df.Location)] or list(df.Location.value_counts().index[:5])
grp = [df[df.Location == c].Humidity3pm.dropna() for c in cities]
print(f"\nCities: {cities}, group sizes: {[len(g) for g in grp]}")
print("Shapiro p per city (5000-sample):", [f"{stats.shapiro(g.sample(min(5000, len(g)), random_state=SEED)).pvalue:.2g}" for g in grp])
lev = stats.levene(*grp); print(f"Levene: W={lev.statistic:.2f}, p={lev.pvalue:.3g} (small p = unequal variances; Kruskal-Wallis is reported as a robust check)")
f = stats.f_oneway(*grp); kw = stats.kruskal(*grp)
report("One-way ANOVA", "all city means of Humidity3pm are equal", "at least one city mean differs", "F", f.statistic, f.pvalue, f"(Kruskal-Wallis p = {kw.pvalue:.3g})")
if f.pvalue < ALPHA: print("\nTukey HSD post-hoc:"); print(stats.tukey_hsd(*grp))

In [ ]:
# ---- Chi-square test of independence: RainToday vs RainTomorrow ----
ct = pd.crosstab(df.RainToday, df.RainTomorrow); chi2, p, dof, exp = stats.chi2_contingency(ct)
cramers = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print("Contingency table:"); display(ct); print(f"Assumption: smallest expected count = {exp.min():.1f} (>= 5 required) -> satisfied: {exp.min() >= 5}")
report("Chi-square test of independence", "RainToday and RainTomorrow are independent", "RainToday and RainTomorrow are associated", "chi2", chi2, p, f"(dof = {dof}; Cramer's V = {cramers:.3f})")
print("P(rain tomorrow | rain today) =", round(ct.loc["Yes", "Yes"] / ct.loc["Yes"].sum(), 3), "| P(rain tomorrow | no rain today) =", round(ct.loc["No", "Yes"] / ct.loc["No"].sum(), 3))

## 12-13. Features, chronological split and leakage-safe pipelines
* **Split by time** (no shuffling): train < 2023, validation = 2023, test ≥ 2024 (if the file ends before 2025, e.g. the old Kaggle copy, the timeline is cut 70/15/15 instead). The future is never used to predict the past.
* Imputation, scaling and one-hot encoding live **inside** a `ColumnTransformer` that is fitted on training rows only.
* `Month` is a categorical feature (one-hot) so December and January stay "close" in meaning without extra code; `RainToday` is today's observation, not tomorrow's.

In [ ]:
NUM_F = NUM
CAT_F = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday", "Month"]
FEATURES = NUM_F + CAT_F                                   # the exact input schema of the saved model
assert not {"RISK_MM", "RainTomorrow", "RainTomorrow_bin", "Date", "Year", "Season"} & set(FEATURES), "leakage / non-input column in FEATURES"

X = df[FEATURES].copy(); X["Month"] = X["Month"].astype(str); y = df["RainTomorrow_bin"]
VAL_START, TEST_START = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-01-01")   # Rattle's weatherAUS.csv runs to 2026
if df.Date.max() < pd.Timestamp("2025-01-01"):      # older Kaggle copy (ends 2017): split the timeline 70/15/15 instead
    VAL_START, TEST_START = df.Date.quantile(0.70), df.Date.quantile(0.85)
tr = df.Date < VAL_START; va = (df.Date >= VAL_START) & (df.Date < TEST_START); te = df.Date >= TEST_START
X_train, y_train, X_val, y_val, X_test, y_test = X[tr], y[tr], X[va], y[va], X[te], y[te]
for nm, m_ in [("train", tr), ("validation", va), ("test", te)]:
    print(f"{nm:10s} rows={m_.sum():6d} | {df.Date[m_].min().date()} -> {df.Date[m_].max().date()} | rain rate={y[m_].mean():.3f}")

def make_prep():
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_F), ("cat", cat, CAT_F)])

# Expensive models (KNN, SVM) are trained on a reproducible random subset of the training rows; their validation/test sets are untouched.
KNN_N, SVM_N = 60000, 15000
models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), None),
    "Decision Tree": (DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, class_weight="balanced", random_state=SEED), None),
    "Random Forest": (RandomForestClassifier(n_estimators=100, max_depth=14, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED), None),
    "KNN": (KNeighborsClassifier(n_neighbors=25, n_jobs=-1), KNN_N),
    "SVM (RBF)": (SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=SEED), SVM_N),
    # Gradient Boosting: the histogram version is much faster than GradientBoostingClassifier on ~100k rows
    "Gradient Boosting": (HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_depth=6, class_weight="balanced", random_state=SEED), None)}
print("Majority-class baseline accuracy on test:", round(1 - y_test.mean(), 4), "-> accuracy alone is misleading; we focus on rain recall / F1")

## 14. Class imbalance (only ~1 day in 4-5 is rainy)
Training data is **never resampled** and the test set is never touched. Instead: `class_weight="balanced"` (LR, tree, forest, SVM) and, for every model, a **decision threshold tuned on the validation year** to maximise the F1 of the "Rain" class. KNN has no class weights, so the tuned threshold is its imbalance remedy.

In [ ]:
def get_scores(pipe, X_):
    return pipe.predict_proba(X_)[:, 1] if hasattr(pipe, "predict_proba") else pipe.decision_function(X_)

def tune_threshold(y_true, s):
    p, r, t = precision_recall_curve(y_true, s); f = 2 * p * r / (p + r + 1e-12)
    return float(t[np.nanargmax(f[:-1])])

def evaluate(y_true, s, thr):
    pred = (s >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred), "Balanced Acc": balanced_accuracy_score(y_true, pred),
            "ROC-AUC": roc_auc_score(y_true, s), "PR-AUC": average_precision_score(y_true, s)}

fitted, scores, rows = {}, {}, []
for name, (clf, n_sub) in models.items():
    idx = X_train.sample(n_sub, random_state=SEED).index if n_sub else X_train.index
    pipe = Pipeline([("prep", make_prep()), ("clf", clf)])
    t0 = time.time(); pipe.fit(X_train.loc[idx], y_train.loc[idx]); fit_s = time.time() - t0
    s_val, s_te = get_scores(pipe, X_val), get_scores(pipe, X_test)
    default_thr = 0.5 if hasattr(pipe, "predict_proba") else 0.0
    thr = tune_threshold(y_val, s_val); m = evaluate(y_test, s_te, thr); m_def = evaluate(y_test, s_te, default_thr)
    rows.append({"Model": name, "Train rows": len(idx), "Fit sec": round(fit_s, 1), "Threshold": thr,
                 "Val F1 (tuned)": f1_score(y_val, (s_val >= thr).astype(int)), **m,
                 "Recall @default thr": m_def["Recall"], "F1 @default thr": m_def["F1"]})
    fitted[name], scores[name] = pipe, s_te
    print(f"{name:20s} done in {fit_s:6.1f}s | test F1={m['F1']:.3f} recall={m['Recall']:.3f} AUC={m['ROC-AUC']:.3f}")

## 15. Model comparison and best-model selection
**Selection criterion (fixed before looking at results):** the model with the highest **F1-score of the "Rain" class on the validation year**, using each model's own tuned threshold. F1 balances precision and recall for the minority class, so accuracy never decides. Test-set Recall and ROC-AUC are shown beside it as a sanity check; the test set is only used to *report* results, never to choose.

In [ ]:
res = pd.DataFrame(rows).set_index("Model"); display(res.round(4))
res.round(5).reset_index().to_csv("model_metrics.csv", index=False)
best_name = res["Val F1 (tuned)"].idxmax()        # selection uses the VALIDATION year only; test is reported, never used to choose
best_pipe, best_thr = fitted[best_name], float(res.loc[best_name, "Threshold"])
print(f"\nBest model (highest validation F1 for the Rain class): {best_name} | threshold = {best_thr:.4f}")
print("Test-set ranking by F1:", list(res.F1.sort_values(ascending=False).index))
print("Test-set ranking by ROC-AUC:", list(res["ROC-AUC"].sort_values(ascending=False).index))

pred_best = (scores[best_name] >= best_thr).astype(int)
print(f"\nClassification report - {best_name} on the 2024+ test set"); print(classification_report(y_test, pred_best, target_names=["No Rain", "Rain"], digits=3))

# charts: metric comparison, ROC curves, confusion matrices
res[["Precision", "Recall", "F1", "ROC-AUC"]].plot.bar(figsize=(11, 4.5), rot=15); plt.ylim(0, 1); plt.title("Model comparison on the chronological test set"); plt.legend(loc="lower right")
save_fig("14_model_comparison")
plt.figure(figsize=(6.5, 5.5))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, scores[name]); plt.plot(fpr, tpr, label=f"{name} (AUC={res.loc[name, 'ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves (test)"); plt.legend(fontsize=8)
save_fig("15_roc_curves")
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, name in zip(ax.ravel(), models):
    cm = confusion_matrix(y_test, (scores[name] >= res.loc[name, "Threshold"]).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a, xticklabels=["No Rain", "Rain"], yticklabels=["No Rain", "Rain"])
    a.set_title(name); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
save_fig("16_confusion_matrices")

# interpretability of the best model, when available
clf_ = best_pipe.named_steps["clf"]; names_ = best_pipe.named_steps["prep"].get_feature_names_out()
imp = getattr(clf_, "feature_importances_", None)
if imp is None and hasattr(clf_, "coef_"): imp = np.abs(clf_.coef_[0])
if imp is not None and len(imp) == len(names_):
    top = pd.Series(imp, index=names_).sort_values().tail(15)
    top.plot.barh(figsize=(8, 5), color="seagreen"); plt.title(f"Top 15 features - {best_name}"); save_fig("17_feature_importance")

## 16-17. Save artifacts
* `cleaned_weather.csv`, `model_metrics.csv` (already written), graphs in `outputs/`.
* `best_rainfall_model.joblib` = the **complete fitted Pipeline** (imputers + encoder + scaler + classifier), trained on the training years only so it matches the reported test metrics.
* `feature_schema.json` = exact input columns, allowed categories, plausible ranges, tuned threshold and three real test rows as GUI examples. The GUIs read this file, so they cannot drift from the model.

In [ ]:
UNITS = {"MinTemp":"°C","MaxTemp":"°C","Rainfall":"mm","Evaporation":"mm","Sunshine":"hours","WindGustSpeed":"km/h","WindSpeed9am":"km/h","WindSpeed3pm":"km/h",
         "Humidity9am":"%","Humidity3pm":"%","Pressure9am":"hPa","Pressure3pm":"hPa","Cloud9am":"oktas (0-9)","Cloud3pm":"oktas (0-9)","Temp9am":"°C","Temp3pm":"°C"}
HARD = {"MinTemp":(-20,55),"MaxTemp":(-20,60),"Rainfall":(0,1000),"Evaporation":(0,100),"Sunshine":(0,24),"WindGustSpeed":(0,250),"WindSpeed9am":(0,250),"WindSpeed3pm":(0,250),
        "Humidity9am":(0,100),"Humidity3pm":(0,100),"Pressure9am":(900,1100),"Pressure3pm":(900,1100),"Cloud9am":(0,9),"Cloud3pm":(0,9),"Temp9am":(-20,60),"Temp3pm":(-20,60)}
def clean(v): return None if (v is None or (isinstance(v, float) and np.isnan(v))) else (float(v) if isinstance(v, (int, float, np.number)) else str(v))
ex_idx = [X_test[y_test == 1].sample(1, random_state=SEED).index[0], X_test[y_test == 0].sample(1, random_state=SEED).index[0], X_test[y_test == 1].sample(1, random_state=SEED + 1).index[0]]
schema = {
    "model_name": best_name, "features": FEATURES, "numeric_features": NUM_F, "categorical_features": CAT_F,
    "numeric": {c: {"unit": UNITS[c], "hard_min": HARD[c][0], "hard_max": HARD[c][1], "train_min": float(X_train[c].min()), "train_max": float(X_train[c].max()), "train_median": float(X_train[c].median())} for c in NUM_F},
    "categorical": {c: sorted(X_train[c].dropna().unique().tolist(), key=(lambda v: int(v)) if c == "Month" else str) for c in CAT_F},
    "threshold": best_thr, "has_predict_proba": bool(hasattr(best_pipe, "predict_proba")),
    "split": {"validation_start": str(pd.Timestamp(VAL_START).date()), "test_start": str(pd.Timestamp(TEST_START).date())},
    "test_metrics": {k: float(v) for k, v in res.loc[best_name, ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].items()},
    "examples": [{"actual_RainTomorrow": "Yes" if y_test.loc[i] else "No", **{c: clean(X_test.loc[i, c]) for c in FEATURES}} for i in ex_idx]}
json.dump(schema, open("feature_schema.json", "w"), indent=2)
joblib.dump(best_pipe, "best_rainfall_model.joblib", compress=3)
print("Saved best_rainfall_model.joblib:", round(os.path.getsize("best_rainfall_model.joblib") / 1e6, 1), "MB | feature_schema.json | model_metrics.csv | cleaned_weather.csv | outputs/:", len(os.listdir("outputs")), "files")
print("Model input columns:", list(best_pipe.feature_names_in_) == FEATURES)

## 17b. Save a report for the Streamlit dashboard
`model_report.json` stores the dataset overview, the selection criterion and every model's confusion matrix, all computed above, so the app can show them without re-running anything.

In [ ]:
report = {
    "selection_criterion": "Highest F1-score of the Rain class on the validation year (threshold tuned on that year); test metrics are reported only.",
    "best_model": best_name,
    "dataset": {"source_file": str(path), "rows_raw": int(len(df_raw)), "rows_used": int(len(df)), "columns_raw": int(df_raw.shape[1]),
                "date_min": str(df.Date.min().date()), "date_max": str(df.Date.max().date()), "locations": int(df.Location.nunique()),
                "rain_tomorrow_rate": float(df.RainTomorrow_bin.mean()), "removed_leaky_columns": leaky,
                "missing_pct": {c: float(round(df[c].isna().mean() * 100, 2)) for c in NUM_F},
                "rows_train": int(tr.sum()), "rows_validation": int(va.sum()), "rows_test": int(te.sum()),
                "test_rain_rate": float(y_test.mean()), "majority_class_accuracy_test": float(1 - y_test.mean())},
    "confusion_matrices": {n: confusion_matrix(y_test, (scores[n] >= res.loc[n, "Threshold"]).astype(int)).tolist() for n in models},
    "correlation_with_target": df[NUM].corrwith(df.RainTomorrow_bin).round(4).to_dict()}
json.dump(report, open("model_report.json", "w"), indent=2)
print("Saved model_report.json")

## 17c. Extra exports for the SkySense app (explainability, calibration, profiles, data)
* **Permutation importance** - how much the test ROC-AUC drops when one input column is shuffled (works for any model; shows how the *model* uses information, **not** physical causation).
* **Score calibration** - among past test days with a given model score, how often did rain really follow? This makes a probability-style score meaningful.
* **Rainy vs dry profile** - median of each measurement before rainy / dry days (training years only).
* `weather_insights.csv.gz` - the cleaned data *without* imputed values, used by the app's Insights pages.

In [ ]:
from sklearn.inspection import permutation_importance
rep = json.load(open("model_report.json"))

# 1) permutation importance of the saved best model (subsample keeps SVM/KNN fast)
sub_ = X_test.sample(min(3000, len(X_test)), random_state=SEED)
pi = permutation_importance(best_pipe, sub_, y_test.loc[sub_.index], scoring="roc_auc", n_repeats=3, random_state=SEED, n_jobs=1)
rep["permutation_importance"] = {"scoring": "drop in test ROC-AUC when the column is shuffled", "rows_used": int(len(sub_)),
    "values": {c: {"mean": float(m), "std": float(s)} for c, m, s in zip(FEATURES, pi.importances_mean, pi.importances_std)}}
display(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).head(8).round(4))

# 2) calibration table (only if the model outputs probabilities)
sc = np.asarray(scores[best_name]); edges = [0, .2, .4, .6, .8, 1.0001]
if hasattr(best_pipe, "predict_proba"):
    b = np.digitize(sc, edges[1:-1])
    rep["calibration"] = [{"low": edges[i], "high": min(edges[i + 1], 1.0), "n": int((b == i).sum()),
                           "rain_rate": float(y_test.values[b == i].mean()) if (b == i).sum() else None} for i in range(5)]
    display(pd.DataFrame(rep["calibration"]).round(3))
else:
    rep["calibration"] = None

# 3) typical values before rainy vs dry days (training years, raw data incl. NaN ignored)
trd = df[tr]
rep["profile"] = {c: {"dry_median": float(trd.loc[trd.RainTomorrow_bin == 0, c].median()), "rain_median": float(trd.loc[trd.RainTomorrow_bin == 1, c].median()),
                      "p1": float(trd[c].quantile(.01)), "p99": float(trd[c].quantile(.99))} for c in NUM_F}

# 4) ROC curve points for every model (downsampled)
rep["roc"] = {}
for n_ in models:
    fpr, tpr, _ = roc_curve(y_test, scores[n_]); ix = np.linspace(0, len(fpr) - 1, min(150, len(fpr))).astype(int)
    rep["roc"][n_] = {"fpr": fpr[ix].round(4).tolist(), "tpr": tpr[ix].round(4).tolist()}
rep["models_info"] = {n_: {"params": {k: str(v) for k, v in models[n_][0].get_params().items() if k in ("C", "max_depth", "n_estimators", "n_neighbors", "max_iter", "learning_rate", "kernel", "min_samples_leaf", "class_weight")},
                           "train_rows": int(KNN_N if n_ == "KNN" else SVM_N if n_ == "SVM (RBF)" else tr.sum())} for n_ in models}
json.dump(rep, open("model_report.json", "w"), indent=2)

# 5) data for the Insights pages (real values, NaN kept, no imputation)
keep = ["Date", "Location"] + NUM + WIND + ["RainToday", "RainTomorrow_bin", "Year", "Month", "Season"]
df[keep].assign(Date=df.Date.dt.strftime("%Y-%m-%d")).to_csv("weather_insights.csv.gz", index=False, compression="gzip")
print("Updated model_report.json and saved weather_insights.csv.gz", round(os.path.getsize("weather_insights.csv.gz") / 1e6, 1), "MB")

## 18. Reload the saved model and verify predictions

In [ ]:
loaded = joblib.load("best_rainfall_model.joblib"); sch = json.load(open("feature_schema.json"))
assert list(loaded.feature_names_in_) == sch["features"], "schema mismatch"
sample = X_test.sample(8, random_state=SEED)
s_loaded = get_scores(loaded, sample); s_orig = get_scores(best_pipe, sample)
assert np.allclose(s_loaded, s_orig), "reloaded model differs from in-memory model"
out = pd.DataFrame({"Location": sample.Location, "Month": sample.Month, "Score": s_loaded.round(3), "Predicted": np.where(s_loaded >= sch["threshold"], "Rain Tomorrow", "No Rain Tomorrow"), "Actual": y_test.loc[sample.index].map({1: "Rain", 0: "No Rain"})})
display(out)
# missing values are handled by the pipeline's imputers (same behaviour the GUI relies on)
sparse_row = pd.DataFrame([{c: np.nan for c in sch["features"]} | {"Location": "Sydney", "Month": "7", "Humidity3pm": 85.0, "Pressure3pm": 1006.0}])
print("Row with only 4 known inputs ->", get_scores(loaded, sparse_row).round(3), "(no error, NaNs imputed from training statistics)")
print("Reload check passed: predictions identical to the in-memory pipeline.")

In [ ]:
# Optional: bundle everything for download (Colab) - works locally too
import shutil
shutil.make_archive("project_artifacts", "zip", ".", "outputs")
try:
    from google.colab import files
    for f in ["best_rainfall_model.joblib", "feature_schema.json", "model_metrics.csv", "model_report.json", "weather_insights.csv.gz", "cleaned_weather.csv", "project_artifacts.zip"]:
        files.download(f)
except ImportError:
    print("Not in Colab - files are already in the current folder.")